# 多记录一个数，真的多知道了一件事吗？

沿用第十三章的和与差，再增加到三个未知数。每次先判断两组输入能否被区分，再计算秩作核对。

依赖 NumPy、Matplotlib。图中为英文，避免中文字体缺失。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
plt.rcParams.update({"font.family": "DejaVu Sans", "font.size": 11})
D = np.array([[1.,1.], [2.,2.]])
S = np.array([[1.,1.], [1.,-1.]])
first = np.array([3.,2.])
second = np.array([4.,1.])


## 1. 加上和的两倍

两条记录是 x1+x2 与 2x1+2x2。先预测它们能否分清 (3,2)、(4,1)。

从第二行减去第一行的两倍。为什么输出写了两个数，却只有一个独立关系？

In [ ]:
print("first records:", D @ first)
print("second records:", D @ second)
U = D.copy()
U[1] -= 2*U[0]
print("elimination:\n", U)
assert np.allclose(D @ first, D @ second)
assert np.allclose(U[1], 0)


## 2. 把第二条改成差

先预测：第一项输出还相同，第二项呢？把两条方程相加，怎样恢复第一个数？

In [ ]:
print("first sum/difference:", S @ first)
print("second sum/difference:", S @ second)
p, q = S @ first
recovered = np.array([(p+q)/2, (p-q)/2])
assert np.allclose(recovered, first)
assert not np.allclose(S @ first, S @ second)
print("ranks, after predicting:", np.linalg.matrix_rank(D), np.linalg.matrix_rank(S))


## 3. 看输出范围，不只是数输出坐标

左图画重复记录的输出，右图画和与差的输出。输入采样相同。

为什么左图只在直线上？为什么右图能遍及平面？采样图不能证明全部可达：请用 (p+q)/2 与 (p-q)/2 的恢复规则说明。

In [ ]:
values = np.linspace(-2,2,9)
inputs = np.array([(x,y) for x in values for y in values]).T
fig, axes = plt.subplots(1,2,figsize=(10,4.5),dpi=160)
for ax, matrix, title, ylabel in [
    (axes[0], D, "Repeated record: rank 1", "Twice the sum"),
    (axes[1], S, "Independent records: rank 2", "Difference")
]:
    outputs = matrix @ inputs
    ax.scatter(*outputs, c=inputs[0], cmap="viridis", s=20)
    for index, color in [(0,"#2878b5"),(1,"#d97922")]:
        column = matrix[:,index]
        ax.annotate("",xy=column,xytext=(0,0),
                    arrowprops={"arrowstyle":"->","color":color,"lw":2})
    ax.set(xlabel="Sum",ylabel=ylabel,title=title,xlim=(-5,5),ylim=(-9,9))
    ax.axhline(0,color="0.7",lw=0.8)
    ax.axvline(0,color="0.7",lw=0.8)
    ax.grid(alpha=0.2)
fig.tight_layout()
plt.show()


## 4. 三个未知数，两次测量

记录 x1+x2=5，x2+x3=7。先选中间的数 t，再写出另外两个。

两组输入 (3,2,5)、(2,3,4) 的差是什么？它对测量有什么影响？

In [ ]:
B = np.array([[1.,1.,0.], [0.,1.,1.]])
direction = np.array([-1.,1.,-1.])
p, q = 5., 7.
t = 3.
candidate = np.array([p-t,t,q-t])
print("input:", candidate, "records:", B @ candidate)
print("invisible change:", direction, "output:", B @ direction)
R = B.copy()
R[0] -= R[1]
print("reduced form:\n", R)
assert np.allclose(B @ candidate, [p,q])
assert np.allclose(B @ direction, 0)
assert np.allclose(R, [[1,0,-1],[0,1,1]])


## 5. 新增第三次测量：先预测，后核对

第一种新增规则是 x1+2x2+x3；第二种是 x1+x2+x3。

先用 direction 测试：哪条记录仍然看不见这次变化？哪条能发现它？再比较两组输入。

In [ ]:
repeat = np.array([1.,2.,1.])
total = np.array([1.,1.,1.])
B_repeat = np.vstack([B,repeat])
B_total = np.vstack([B,total])
for name, measurement, matrix in [
    ("repeated",repeat,B_repeat), ("total",total,B_total)
]:
    print(name, "change detected:", measurement @ direction)
    print("first records:", matrix @ np.array([3.,2.,5.]))
    print("second records:", matrix @ np.array([2.,3.,4.]))
    print("rank:", np.linalg.matrix_rank(matrix))
assert np.isclose(repeat @ direction,0)
assert not np.isclose(total @ direction,0)
assert np.linalg.matrix_rank(B_repeat)==2
assert np.linalg.matrix_rank(B_total)==3
# Recover from pair sums p, q and the total s.
s = 10.
middle = p+q-s
recovered = np.array([p-middle,middle,q-middle])
assert np.allclose(B_total @ recovered,[p,q,s])
print("recovered:",recovered)


## 6. 重复记录若写错了，会怎样？

和为 5 时，两倍的和记录成 11。先手算消元后的最后一条方程。

这不是自由变量变多，而是没有输入符合记录。

In [ ]:
bad_target = np.array([5.,11.])
residual = bad_target[1]-2*bad_target[0]
print("last equation: 0 =",residual)
relation = np.array([-2.,1.])
assert np.allclose(relation @ D,0)
assert np.isclose(relation @ bad_target,residual)
assert not np.isclose(residual,0)


## 7. 主元与自由变量的计数

先在纸上填表：D、S、B、B_repeat、B_total 各有几个主元，几个自由变量？

下面数值秩只核对本实验精确构造的矩阵；接近相关的浮点数据受容差影响。解释定理仍要用消元，不用函数返回值代替证明。

In [ ]:
for name, matrix in [("D",D),("S",S),("B",B),
                     ("B_repeat",B_repeat),("B_total",B_total)]:
    n = matrix.shape[1]
    rank = np.linalg.matrix_rank(matrix)
    print(name,"input coordinates:",n,"rank:",rank,"free variables:",n-rank)


## 重建

1. 为什么两条记录可能只有一个主元？
2. 给 B 增加只记录 x2 的第三条测量，它能分清原来不可见的变化吗？先手算。
3. 为什么秩加零空间维数等于输入坐标数，而不是输出坐标数？
4. 为什么重复记录可以导致目标不相容，却不能消除原来的自由变量？